In [ ]:
import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "127.0.0.1"

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import re

matplotlib.rcParams["figure.dpi"] = 100
matplotlib.rcParams["font.size"] = 11

In [ ]:
spark = SparkSession.builder \
    .appName("clickstream-recommender-notebook") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.ui.enabled", "false") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")

In [ ]:
base = Path(".").resolve().parent
cleaned_path = base / "output" / "cleaned.parquet"
dataset_path = base / "dataset" / "e-shop clothing 2008.csv"

if cleaned_path.exists() and any(cleaned_path.glob("*.parquet")):
    df = spark.read.parquet(str(cleaned_path))
    print("Loaded from cleaned.parquet")
else:
    df = spark.read.csv(str(dataset_path), header=True, inferSchema=True, sep=";")
    for col in df.columns:
        df = df.withColumnRenamed(col, re.sub(r'[ ()/]+', '_', col).strip('_').lower())
    df = df.withColumn("date", F.to_date(F.concat_ws("-", F.col("year"), F.col("month"), F.col("day")), "yyyy-M-d")).drop("year", "month", "day")
    for c in ["country", "page_1_main_category", "colour", "location", "model_photography"]:
        df = df.withColumn(c, F.col(c).cast(StringType()))
    df = df.withColumn("price_2", F.col("price_2") == 1)
    print("Loaded from raw CSV")

df.printSchema()

In [ ]:
interactions = (
    df.groupBy(
        F.col("session_id").cast("string").alias("session_id"),
        "page_2_clothing_model"
    )
    .agg(F.count("*").alias("click_count"))
)

interactions.cache()

n_sessions = interactions.select("session_id").distinct().count()
n_items = interactions.select("page_2_clothing_model").distinct().count()
n_total = interactions.count()
sparsity = 1.0 - (n_total / (n_sessions * n_items))

print(f"Interaction matrix: {n_sessions} sessions x {n_items} products")
print(f"Total possible pairs: {n_sessions * n_items:,}")
print(f"Non-zero entries: {n_total:,}")
print(f"Matrix density: {1 - sparsity:.4f}  |  Sparsity: {sparsity:.4f}")
interactions.show(5)

In [ ]:
click_pd = interactions.groupBy("click_count").count().orderBy("click_count").toPandas()

product_pop = (
    interactions.groupBy("page_2_clothing_model")
    .agg(F.sum("click_count").alias("total_clicks"), F.count("*").alias("n_sessions"))
    .orderBy(F.col("total_clicks").desc())
    .limit(20)
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(click_pd["click_count"].clip(upper=10).values,
            click_pd["count"].values, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Click count per (session, product) pair")
axes[0].set_ylabel("Number of pairs")
axes[0].set_title("Distribution of interaction strengths")
axes[0].set_yscale("log")

axes[1].barh(range(len(product_pop)), product_pop["total_clicks"], color="steelblue")
axes[1].set_yticks(range(len(product_pop)))
axes[1].set_yticklabels(product_pop["page_2_clothing_model"], fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel("Total clicks across all sessions")
axes[1].set_title("Top 20 most viewed products")

plt.tight_layout()
out_dir = base / "output"
out_dir.mkdir(exist_ok=True)
plt.savefig(str(out_dir / "recommender_interaction_dist.png"), bbox_inches="tight")
plt.show()

In [ ]:
session_model = StringIndexer(inputCol="session_id", outputCol="user_idx").fit(interactions)
interactions = session_model.transform(interactions)

product_model = StringIndexer(inputCol="page_2_clothing_model", outputCol="item_idx").fit(interactions)
interactions = product_model.transform(interactions)

interactions = interactions \
    .withColumn("user_idx", F.col("user_idx").cast("int")) \
    .withColumn("item_idx", F.col("item_idx").cast("int"))

train, test = interactions.randomSplit([0.8, 0.2], seed=42)
train.cache()

print(f"Train: {train.count()} rows  |  Test: {test.count()} rows")
print(f"Products indexed: {len(product_model.labels)}")
print(f"Sample product labels: {product_model.labels[:10]}")

In [ ]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="click_count",
    predictionCol="prediction"
)

configs = [
    {"rank": 5,  "regParam": 0.1},
    {"rank": 10, "regParam": 0.1},
    {"rank": 20, "regParam": 0.1},
    {"rank": 5,  "regParam": 0.01},
    {"rank": 10, "regParam": 0.01},
    {"rank": 20, "regParam": 0.01},
]

best_rmse = float("inf")
best_model = None
best_config = None
results = []

for cfg in configs:
    als = ALS(
        rank=cfg["rank"],
        regParam=cfg["regParam"],
        maxIter=10,
        userCol="user_idx",
        itemCol="item_idx",
        ratingCol="click_count",
        implicitPrefs=True,
        coldStartStrategy="drop",
        seed=42
    )
    model = als.fit(train)
    preds = model.transform(test)
    rmse = evaluator.evaluate(preds)
    results.append({"rank": cfg["rank"], "regParam": cfg["regParam"], "rmse": rmse})
    print(f"rank={cfg['rank']:2d}  regParam={cfg['regParam']:.2f}  RMSE={rmse:.4f}")
    if rmse < best_rmse:
        best_rmse = rmse
        best_model = model
        best_config = cfg

print(f"\nBest: rank={best_config['rank']}, regParam={best_config['regParam']}, RMSE={best_rmse:.4f}")

In [ ]:
results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["steelblue", "darkorange"]
for i, reg in enumerate(sorted(results_df["regParam"].unique())):
    subset = results_df[results_df["regParam"] == reg].sort_values("rank")
    ax.plot(subset["rank"], subset["rmse"],
            marker="o", linewidth=2, color=colors[i], label=f"regParam = {reg}")
    for _, row in subset.iterrows():
        ax.annotate(f"{row['rmse']:.3f}", (row["rank"], row["rmse"]),
                    textcoords="offset points", xytext=(4, 4), fontsize=9)

ax.set_xlabel("Rank (number of latent factors)")
ax.set_ylabel("RMSE on test set")
ax.set_title("ALS hyperparameter search: RMSE by rank and regularization")
ax.set_xticks([5, 10, 20])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(base / "output" / "recommender_hyperparams.png"), bbox_inches="tight")
plt.show()

In [ ]:
user_recs = best_model.recommendForAllUsers(10)

item_labels_df = spark.createDataFrame(
    [(i, product_model.labels[i]) for i in range(len(product_model.labels))],
    ["item_idx", "page_2_clothing_model"]
)

session_labels_df = spark.createDataFrame(
    [(i, str(session_model.labels[i])) for i in range(len(session_model.labels))],
    ["user_idx", "session_id"]
)

recommendations = (
    user_recs
    .select("user_idx", F.explode("recommendations").alias("rec"))
    .select(
        "user_idx",
        F.col("rec.item_idx").alias("item_idx"),
        F.col("rec.rating").alias("confidence_score")
    )
    .join(item_labels_df, on="item_idx", how="left")
    .join(session_labels_df, on="user_idx", how="left")
    .select("session_id", "page_2_clothing_model", "confidence_score")
    .orderBy("session_id", F.col("confidence_score").desc())
)

n_covered = recommendations.select("session_id").distinct().count()
n_items_rec = recommendations.select("page_2_clothing_model").distinct().count()

print(f"Sessions with recommendations: {n_covered:,} / {n_sessions:,}")
print(f"Distinct items recommended: {n_items_rec} / {n_items}")
print(f"Catalogue coverage: {n_items_rec / n_items:.2%}")

In [ ]:
item_freq = (
    recommendations.groupBy("page_2_clothing_model")
    .count()
    .orderBy(F.col("count").desc())
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

top20 = item_freq.head(20)
axes[0].barh(range(len(top20)), top20["count"], color="steelblue")
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(top20["page_2_clothing_model"], fontsize=9)
axes[0].invert_yaxis()
axes[0].set_xlabel("Number of sessions recommended")
axes[0].set_title("Top 20 most recommended products")

axes[1].hist(item_freq["count"], bins=30, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Number of sessions a product is recommended to")
axes[1].set_ylabel("Number of products")
axes[1].set_title("Distribution of recommendation frequency across catalogue")

plt.tight_layout()
plt.savefig(str(base / "output" / "recommender_coverage.png"), bbox_inches="tight")
plt.show()

In [ ]:
sample_sessions = (
    interactions.select("session_id")
    .distinct()
    .limit(5)
    .toPandas()["session_id"].tolist()
)

print("Sample recommendations for selected sessions:")
print("-" * 60)
for sid in sample_sessions:
    history = (
        interactions.filter(F.col("session_id") == sid)
        .select("page_2_clothing_model", "click_count")
        .orderBy(F.col("click_count").desc())
        .toPandas()
    )
    recs_for_session = (
        recommendations.filter(F.col("session_id") == sid)
        .orderBy(F.col("confidence_score").desc())
        .toPandas()
    )
    print(f"Session {sid}:")
    print(f"  Viewed: {list(history['page_2_clothing_model'])}")
    print(f"  Recommended: {list(recs_for_session['page_2_clothing_model'])}")
    print()

In [ ]:
output_path = base / "output" / "recommendations.parquet"
recommendations.write.parquet(str(output_path), mode="overwrite")
print(f"Recommendations saved to {output_path}")
print(f"Total recommendation rows: {recommendations.count():,}")

In [ ]:
spark.stop()
print("Spark session stopped.")